In [34]:
# Librerie minime per test API
import requests
from uuid import uuid4
from pprint import pprint

In [35]:
# Configurazione endpoint
# Docker: docker run -d --rm --name deployer -p 8080:8080 deployer:latest
# Se hai appena modificato app.py, fai prima:
# docker build -t deployer:latest .

base_url_docker = "http://localhost:8080"
base_url_local = "http://localhost:8000"

# Scegli l'ambiente da testare
base_url = base_url_local

In [36]:
# Endpoint principali dell'API
url_health = f"{base_url}/health"
url_ready = f"{base_url}/ready"
url_predict = f"{base_url}/predict/"

In [37]:
# Payload valido (esempio policyholder)
policyholder_dict = {
"VehPower":6,
"VehAge":10,
"DrivAge":18,
"Density":400,
"BonusMalus":50,
"VehBrand":"B12",
"VehGas":"Regular",
"Region":"R72",
"Area":"A"
}

In [38]:
# 1) Liveness e Readiness check
health_response = requests.get(url_health, timeout=10)
ready_response = requests.get(url_ready, timeout=10)

print("/health ->", health_response.status_code)
if health_response.headers.get("content-type", "").startswith("application/json"):
    pprint(health_response.json())
else:
    print(health_response.text)

print("/ready  ->", ready_response.status_code)
if ready_response.headers.get("content-type", "").startswith("application/json"):
    pprint(ready_response.json())
else:
    print(ready_response.text)

# Fallback diagnostico: alcune immagini vecchie non espongono /health e /ready.
if health_response.status_code == 404 and ready_response.status_code == 404:
    root_response = requests.get(base_url + "/", timeout=10)
    print("/      ->", root_response.status_code)
    if root_response.headers.get("content-type", "").startswith("application/json"):
        pprint(root_response.json())
    else:
        print(root_response.text)
    print("Nota: se vuoi i nuovi endpoint, ricrea e rilancia l'immagine Docker aggiornata.")

/health -> 200
{'status': 'ok'}
/ready  -> 200
{'status': 'ready'}


In [39]:
# 2) Prediction con request id (utile per tracing nei log)
headers = {"x-request-id": str(uuid4())}
response = requests.post(url_predict, json=policyholder_dict, headers=headers, timeout=20)

print("/predict ->", response.status_code)
if response.headers.get("content-type", "").startswith("application/json"):
    pprint(response.json())
else:
    print(response.text)

/predict -> 200
{'Frequency': 0.6384685103649296,
 'Pure_Premium': 1363.0526530730017,
 'Severity': 2134.878433227539}


In [40]:
# 3) Esempio errore input: validazione FastAPI/Pydantic
invalid_policyholder = dict(policyholder_dict)
invalid_policyholder["VehPower"] = 99  # fuori range [1, 20]

bad_response = requests.post(url_predict, json=invalid_policyholder, timeout=20)
print("/predict (invalid) ->", bad_response.status_code)
if bad_response.headers.get("content-type", "").startswith("application/json"):
    pprint(bad_response.json())
else:
    print(bad_response.text)

/predict (invalid) -> 422
{'detail': [{'ctx': {'le': 20},
             'input': 99,
             'loc': ['body', 'VehPower'],
             'msg': 'Input should be less than or equal to 20',
             'type': 'less_than_equal'}]}
